# Text matching / retrieval · review
Lima pendekatan: TF-IDF, BM25, E5 chunk retrieval, RRF hybrid, hybrid+cross-encoder rerank. Ditambah kalibrasi skor untuk keputusan match/tidak match.

Default [IndoSum](https://www.kaggle.com/datasets/linkgish/indosum): kalimat ringkasan menjadi kueri proxy, artikel asal menjadi satu dokumen relevan. Ini benchmark turunan yang memihak overlap kata, bukan evaluasi kueri manusia atau relevance judgment lengkap. Untuk data nyata, isi dua CSV pada konfigurasi. Corpus diketahui saat indexing; label kueri test tidak dipakai memilih metode.

## 00 · Konfigurasi

Ubah sumber/kolom/ukuran sampel di sini. Tidak ada grafik otomatis.

In [1]:
SEED,TOP=42,15
RESULT_NAME='hasil_retrieval_review'
N_DOCS,N_QUERIES,EDA_N=2000,400,3000
RERANK_TOP,CHUNK_WORDS,MAX_CHUNKS=20,120,6
CORPUS_CSV,QUERIES_CSV=None,None

## 01 · Lingkungan & preprocessing teknis

Safe cleaning: encoding, NFC, HTML entity, invisible, whitespace. Pertahankan angka, negasi, case, entitas dan bahasa. Tanpa stemming/stopword removal pada encoder.

In [2]:
import os,sys,subprocess,time,json,re,html,unicodedata,random,gc
from pathlib import Path
from collections import Counter
assert Path('/kaggle/input').exists(),'Jalankan di Kaggle.'
os.environ.update(USE_TF='0',USE_FLAX='0',TOKENIZERS_PARALLELISM='false',HF_HUB_DISABLE_PROGRESS_BARS='1')
subprocess.run([sys.executable,'-m','pip','install','-q','ftfy>=6.3,<7','rank-bm25>=0.2,<1','sentencepiece>=0.2,<1','langid==1.1.6'],check=True)
import numpy as np,pandas as pd,torch,ftfy
from IPython.display import display
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.metrics import f1_score,classification_report,confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import Ridge,LogisticRegression
from transformers import AutoTokenizer,AutoModel,AutoModelForSequenceClassification
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.set_num_threads(2)
assert torch.cuda.is_available(),'Aktifkan GPU Kaggle.'
OUT=Path('/kaggle/working')/RESULT_NAME;OUT.mkdir(exist_ok=True)
pd.set_option('display.max_columns',None,'display.max_colwidth',120)
START=time.perf_counter()

In [3]:
def show(title,table):
    print('\n'+title)
    display((table.to_frame() if isinstance(table,pd.Series) else table).round(4))

def distribution(values):
    n=values.value_counts(dropna=False)
    return pd.DataFrame({'n':n,'%':100*n/(len(values) or np.nan)})

def clean(value):
    if not isinstance(value,str): return ''
    repaired=ftfy.fix_text(value,unescape_html=False,uncurl_quotes=False,fix_latin_ligatures=False,
        fix_character_width=False,remove_control_chars=False,normalization=None)
    text=unicodedata.normalize('NFC',html.unescape(repaired))
    text=text.translate(dict.fromkeys(map(ord,'\ufeff\u200b\u2060\x00'),None))
    text=re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]',' ',text)
    return re.sub(r'\s+',' ',text).strip()

def key(value): return clean(value).casefold()
def cap(frame,n): return frame.sample(min(n,len(frame)),random_state=SEED).reset_index(drop=True) if n else frame.reset_index(drop=True)
def locate(folder,repo):
    matches=list(Path('/kaggle/input').glob('**/models/'+folder+'/config.json'))
    return str(matches[0].parent) if matches else repo
NOISE={'url':r'https?://|www\.', 'mention':r'@\w+','hashtag':r'#\w+','angka':r'\d',
 'emoji/simbol':r'[\U0001F300-\U0001FAFF\u2600-\u27BF]','huruf_ulang':r'([^\W\d_])\1{2,}',
 'baca_ulang':r'[!?.,]{2,}','html':r'</?[a-zA-Z][^>]*>','encoding':r'\ufffd|Ã.|Â.',
 'non_ascii':r'[^\x00-\x7f]','spasi_ulang':r' {2,}','invisible':r'[\u200b\ufeff]',
 'KAPITAL':r'(?-i:\b[A-Z]{3,}\b)','huruf_angka':r'\b(?=\w*[a-z])(?=\w*\d)\w+\b',
 'slang_contoh':r'\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b',
 'negasi':r'\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|not|never)\b'}

In [4]:
def text_eda(values,title):
    valid=values.where(values.map(lambda v:isinstance(v,str)),'').fillna('')
    show(title+' kualitas',pd.Series({'n':len(values),'null':values.isna().sum(),'kosong/spasi':valid.str.strip().eq('').sum(),
        'unik_terisi':valid[valid.str.strip().ne('')].nunique(),'duplikat_ekstra_terisi':valid[valid.str.strip().ne('')].duplicated().sum()}))
    show(title+' length',pd.DataFrame({'karakter':valid.str.len(),'kata':valid.str.split().str.len()}).describe(percentiles=[.01,.5,.9,.95,.99]).T)
    sample=valid[valid.str.strip().ne('')].sample(min(EDA_N,valid.str.strip().ne('').sum()),random_state=SEED)
    tokens=sample.str.casefold().str.findall(r'[^\W\d_]+')
    count=pd.Series(Counter(w for seq in tokens for w in seq),dtype='int64').sort_values(ascending=False)
    show(title+' vocabulary sampel',pd.Series({'n_sample':len(sample),'token':count.sum(),'unik':len(count),
        'hapax':count.eq(1).sum(),'hapax_%_tipe':100*count.eq(1).mean(),'hapax_%_token':100*count.eq(1).sum()/(count.sum() or np.nan)}))
    show(title+' top words',count.head(TOP).rename('n').to_frame().assign(pct=count.head(TOP)/(count.sum() or np.nan)*100))
    for size in [2,3]:
        grams=pd.Series(Counter(' '.join(seq[i:i+size]) for seq in tokens for i in range(len(seq)-size+1)),dtype='int64').sort_values(ascending=False)
        show(title+f' top {size}-gram',grams.head(TOP))
    flags=pd.DataFrame({name:valid.str.count(regex,flags=re.I).gt(0) for name,regex in NOISE.items()})
    show(title+' noise · % semua baris, dapat overlap',pd.DataFrame({'n':flags.sum(),'%':flags.mean()*100}))
    terms=count.index.to_series().astype('string')
    weird=terms.str.len().ge(20)|terms.str.count(r'([^\W\d_])\1{2,}').gt(0)
    show(title+' kandidat kata aneh, bukan pasti typo',count[weird].head(TOP))
    from langid.langid import LanguageIdentifier,model
    from threadpoolctl import threadpool_limits
    detector=LanguageIdentifier.from_modelstring(model,norm_probs=True)
    lang_sample=sample.sample(min(150,len(sample)),random_state=SEED)
    with threadpool_limits(limits=1):
        lang=lang_sample.map(lambda s:detector.classify(s[:3000]) if len(s.split())>=5 else ('tidak_pasti',np.nan))
    show(title+' estimasi bahasa · % sampel, bukan gold',distribution(lang.map(lambda p:p[0] if p[1]>=.8 else 'tidak_pasti')))

In [5]:
_encoder=None;_encoder_tok=None
@torch.inference_mode()
def embed(texts,prefix='query: '):
    global _encoder,_encoder_tok
    if _encoder is None:
        path=locate('e5','intfloat/multilingual-e5-base')
        _encoder_tok=AutoTokenizer.from_pretrained(path)
        _encoder=AutoModel.from_pretrained(path).cuda().eval()
    texts=list(texts);output=[]
    for start in range(0,len(texts),32):
        batch=_encoder_tok([prefix+t for t in texts[start:start+32]],padding=True,truncation=True,max_length=256,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):hidden=_encoder(**batch).last_hidden_state
        mask=batch.attention_mask.unsqueeze(-1)
        pooled=(hidden.float()*mask).sum(1)/mask.sum(1).clamp(min=1)
        output.append(torch.nn.functional.normalize(pooled,dim=1).cpu().numpy())
    return np.concatenate(output)

def release_encoder():
    global _encoder,_encoder_tok
    _encoder=None;_encoder_tok=None;gc.collect();torch.cuda.empty_cache()

In [6]:
@torch.inference_mode()
def pair_logits(repo,frame,folder=None,max_len=256):
    path=locate(folder,repo) if folder else repo
    tok=AutoTokenizer.from_pretrained(path);model=AutoModelForSequenceClassification.from_pretrained(path).cuda().eval()
    mapping={str(k).lower():int(v) for k,v in model.config.label2id.items()};output=[]
    for start in range(0,len(frame),32):
        batch=frame.iloc[start:start+32]
        enc=tok(batch.a.tolist(),batch.b.tolist(),padding=True,truncation=True,max_length=max_len,return_tensors='pt').to('cuda')
        with torch.autocast('cuda',dtype=torch.float16):logits=model(**enc).logits
        output.append(logits.float().cpu().numpy())
    del model;gc.collect();torch.cuda.empty_cache()
    return np.concatenate(output),mapping

## 02 · Corpus & kueri

CSV opsional: corpus kolom doc_id,text; queries kolom query,relevant_ids (JSON list doc_id). Multi-relevant didukung metrik. Default membuang artikel identik, bukan membuang isi volume tanpa pelaporan.

In [7]:
if CORPUS_CSV and QUERIES_CSV:
    corpus=pd.read_csv(CORPUS_CSV);queries=pd.read_csv(QUERIES_CSV)
    assert {'doc_id','text'}.issubset(corpus) and {'query','relevant_ids'}.issubset(queries)
    corpus['text']=corpus.text.map(clean);assert corpus.doc_id.astype(str).is_unique
    lookup={str(x):i for i,x in enumerate(corpus.doc_id)}
    gold=[{lookup[str(x)] for x in json.loads(v)} for v in queries.relevant_ids]
    queries['query']=queries['query'].map(clean)
else:
    path=next(Path('/kaggle/input').rglob('test.01.jsonl'));raw=pd.read_json(path,lines=True)
    corpus=raw.copy();corpus['text']=corpus.paragraphs.map(lambda p:' '.join(' '.join(s) for para in p for s in para)).map(clean)
    corpus['query']=corpus.summary.map(lambda seq:' '.join(seq[0]) if seq else '').map(clean)
    print('Artikel duplicate ekstra:',corpus.text.map(key).duplicated().sum())
    corpus=cap(corpus.loc[corpus.text.ne('')&corpus['query'].str.split().str.len().ge(5)].drop_duplicates('text'),N_DOCS)
    corpus['doc_id']=np.arange(len(corpus));queries=cap(corpus[['doc_id','query']],N_QUERIES)
    gold=[{int(x)} for x in queries.doc_id]
assert all(gold) and queries['query'].ne('').all() and corpus.text.ne('').all()
query_groups=queries['query'].map(key)
v,t=next(GroupShuffleSplit(n_splits=1,test_size=.6,random_state=SEED).split(queries,groups=query_groups))
assert set(query_groups.iloc[v]).isdisjoint(query_groups.iloc[t])
print('Corpus:',len(corpus),'| validation/test queries:',len(v),len(t))

Artikel duplicate ekstra: 0
Corpus: 2000 | validation/test queries: 160 240


## 03 · EDA retrieval wajib

Ukuran index, panjang dokumen/kueri, vocabulary/noise, gold per kueri, duplikat kueri, chunk coverage dan lexical shortcut.

In [8]:
text_eda(corpus.text,'Dokumen');text_eda(queries['query'],'Kueri')
show('Jumlah gold per kueri',distribution(pd.Series(list(map(len,gold)))))
print('Kueri duplicate:',queries['query'].map(key).duplicated().sum())
overlap=[len(set(q.casefold().split())&set(corpus.text.iloc[min(g)].casefold().split()))/max(1,len(set(q.casefold().split()))) for q,g in zip(queries['query'],gold)]
show('Porsi kata kueri muncul pada salah satu gold · shortcut audit',pd.Series(overlap).describe())


Dokumen kualitas


,0
n,2000
null,0
kosong/spasi,0
unik_terisi,2000
duplikat_ekstra_terisi,0



Dokumen length


,count,mean,std,min,1%,50%,90%,95%,99%,max
karakter,2000.0,2077.9205,809.9024,401.0,691.98,1946.0,3105.4,3621.05,4624.03,6978.0
kata,2000.0,339.2405,128.8625,68.0,115.98,320.0,502.1,579.00,739.03,1118.0



Dokumen vocabulary sampel


,0
n_sample,2000.0000
token,558642.0000
unik,33288.0000
hapax,13407.0000
hapax_%_tipe,40.2758
hapax_%_token,2.3999


... (30 outputs left out)


Jumlah gold per kueri


,n,%
1,400,100.0


Kueri duplicate: 0

Porsi kata kueri muncul pada salah satu gold · shortcut audit


,0
count,400.0000
mean,0.9721
std,0.0864
min,0.2632
25%,1.0000
50%,1.0000
75%,1.0000
max,1.0000


## 04 · Metrik ranking & sanity check

Recall@k memakai jumlah relevant ID, bukan hit rate saja. NDCG memakai binary relevance dengan IDCG menurut jumlah gold. Kandidat yang tidak berisi gold tidak dapat diperbaiki oleh reranker.

In [9]:
def query_metrics(row,truth):
    hits=np.array([int(int(x) in truth) for x in row]);positions=np.flatnonzero(hits[:10])
    result={'MRR@10':1/(positions[0]+1) if len(positions) else 0.}
    for k in [1,5,10]:result[f'Recall@{k}']=hits[:k].sum()/len(truth)
    gain=(hits[:10]/np.log2(np.arange(min(10,len(row)))+2)).sum()
    ideal=(1/np.log2(np.arange(min(10,len(truth)))+2)).sum()
    result['NDCG@10']=gain/ideal
    return result
def ranking_metrics(rank,ids):
    return pd.DataFrame([query_metrics(rank[i],gold[i]) for i in ids]).mean().to_dict()
assert query_metrics(np.array([1,0,2,3,4]),{1,2})['Recall@1']==.5
assert query_metrics(np.array([1,0,2,3,4]),{1,2})['MRR@10']==1
def rank_scores(scores,n=50):return np.argsort(-scores,axis=1,kind='stable')[:,:min(n,scores.shape[1])]

## 05 · TF-IDF, BM25 & chunk E5

Lexical index memakai semua dokumen kandidat, bukan label kueri. Encoder menjaga case/angka. Maksimal MAX_CHUNKS per dokumen; tail yang tidak dibaca dilaporkan. Similarity kueri-dokumen adalah max similarity chunk.

In [10]:
tokenize=lambda s:re.findall(r'[^\W_]+',s.casefold())
tf=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=100000).fit(corpus.text)
tf_scores=(tf.transform(queries['query'])@tf.transform(corpus.text).T).toarray()
from rank_bm25 import BM25Okapi
bm=BM25Okapi(corpus.text.map(tokenize).tolist())
bm_scores=np.stack([bm.get_scores(tokenize(q)) for q in queries['query']])
ranks={'01_tfidf':rank_scores(tf_scores),'02_bm25':rank_scores(bm_scores)}
chunks=[];owners=[]
for i,text in enumerate(corpus.text):
    words=text.split()
    for start in list(range(0,len(words),CHUNK_WORDS))[:MAX_CHUNKS]:chunks.append(' '.join(words[start:start+CHUNK_WORDS]));owners.append(i)
owners=np.asarray(owners);chunk_matrix=embed(chunks,prefix='passage: ');query_matrix=embed(queries['query'],prefix='query: ')
chunk_scores=query_matrix@chunk_matrix.T
dense_scores=np.full((len(queries),len(corpus)),-np.inf,dtype='float32')
for i,row in enumerate(chunk_scores):np.maximum.at(dense_scores[i],owners,row)
ranks['03_e5_chunks']=rank_scores(dense_scores)
release_encoder()
print('Chunks:',len(chunks),'| dokumen dengan tail tidak dibaca (%):',corpus.text.str.split().str.len().gt(CHUNK_WORDS*MAX_CHUNKS).mean()*100)

Chunks: 6587 | dokumen dengan tail tidak dibaca (%): 1.25


## 06 · Hybrid dipilih validation & rerank

RRF alpha dipilih dari validation NDCG@10. Rerank hanya top-k kandidat hybrid, memakai chunk kandidat yang paling dekat menurut E5. Test labels belum dipakai.

In [11]:
def rrf(alpha):
    score=np.zeros((len(queries),len(corpus)))
    for weight,rank in [(1-alpha,ranks['02_bm25']),(alpha,ranks['03_e5_chunks'])]:
        for i,row in enumerate(rank):score[i,row]+=weight/(60+np.arange(len(row))+1)
    return rank_scores(score)
alphas=[0,.25,.5,.75,1]
alpha=max(alphas,key=lambda a:ranking_metrics(rrf(a),v)['NDCG@10'])
ranks['04_hybrid']=rrf(alpha)
pair_rows=[];candidate_ids=[]
for i,row in enumerate(ranks['04_hybrid']):
    for doc in row[:RERANK_TOP]:
        positions=np.flatnonzero(owners==doc);best=positions[np.argmax(chunk_scores[i,positions])]
        pair_rows.append({'a':queries['query'].iloc[i],'b':chunks[best]});candidate_ids.append((i,int(doc)))
logits,_=pair_logits('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1',pd.DataFrame(pair_rows),'reranker',max_len=384)
assert logits.shape[1]==1,'Reranker output bukan satu relevance logit.'
raw_scores=logits[:,0].reshape(len(queries),-1)
reranked=[]
for i,row in enumerate(ranks['04_hybrid']):
    head=row[:RERANK_TOP];order=np.argsort(-raw_scores[i],kind='stable')
    reranked.append(np.r_[head[order],row[RERANK_TOP:]])
ranks['05_hybrid_rerank']=np.asarray(reranked)
dev=pd.DataFrame([{'method':name,**ranking_metrics(rank,v)} for name,rank in ranks.items()]).sort_values('NDCG@10',ascending=False)
show('Validation ranking',dev);chosen=dev.iloc[0]['method']
(OUT/'selection.json').write_text(json.dumps({'method':chosen,'alpha':alpha,'rerank_top':RERANK_TOP},indent=2))


Validation ranking


,method,MRR@10,Recall@1,Recall@5,Recall@10,NDCG@10
1,02_bm25,1.0000,1.0000,1.0000,1.0000,1.0000
3,04_hybrid,1.0000,1.0000,1.0000,1.0000,1.0000
4,05_hybrid_rerank,0.9958,0.9938,1.0000,1.0000,0.9969
0,01_tfidf,0.9794,0.9688,0.9875,1.0000,0.9843
2,03_e5_chunks,0.9641,0.9500,0.9812,0.9812,0.9685


59

## 07 · Binary text matching · skor reranker

Kalibrasi satu logit reranker dengan LR, group cross-fit per kueri pada validation. Threshold dari OOF validation. Negatif adalah kandidat non-gold; pada benchmark relevansi tak lengkap, sebagian bisa sebenarnya relevan. Metrik kandidat bukan metrik seluruh corpus.

In [12]:
from sklearn.model_selection import GroupKFold
candidate_target=np.array([int(doc in gold[q]) for q,doc in candidate_ids])
candidate_query=np.array([q for q,doc in candidate_ids])
flat=raw_scores.ravel()[:,None];dev_mask=np.isin(candidate_query,v);test_mask=np.isin(candidate_query,t)
assert len(np.unique(candidate_target[dev_mask]))==2,'Validation candidates perlu positif dan negatif.'
oof=np.zeros(dev_mask.sum());dev_x,dev_y=flat[dev_mask],candidate_target[dev_mask]
for a,b in GroupKFold(3).split(dev_x,groups=candidate_query[dev_mask]):
    cal=LogisticRegression().fit(dev_x[a],dev_y[a]);oof[b]=cal.predict_proba(dev_x[b])[:,1]
cut=max(np.linspace(.05,.95,37),key=lambda c:f1_score(dev_y,oof>=c,zero_division=0))
cal=LogisticRegression().fit(dev_x,dev_y);match_prob=cal.predict_proba(flat[test_mask])[:,1]
show('Matching test candidates',pd.DataFrame(classification_report(candidate_target[test_mask],match_prob>=cut,output_dict=True,zero_division=0)).T)
(OUT/'matching_policy.json').write_text(json.dumps({'threshold':float(cut),'coef':cal.coef_.tolist(),'intercept':cal.intercept_.tolist()}))


Matching test candidates


,precision,recall,f1-score,support
0,0.9961,0.9965,0.9963,4561.0000
1,0.9325,0.9247,0.9286,239.0000
accuracy,0.9929,0.9929,0.9929,0.9929
macro avg,0.9643,0.9606,0.9624,4800.0000
weighted avg,0.9929,0.9929,0.9929,4800.0000


87

## 08 · Holdout ranking & kelemahan

Laporan per-kueri, category jika ada, answer-missing-in-candidates, dan contoh gagal. Pilihan validation dipertahankan walaupun test metode lain lebih tinggi.

In [13]:
board=pd.DataFrame([{'method':name,**ranking_metrics(rank,t)} for name,rank in ranks.items()])
show('Test ranking',board);board.to_csv(OUT/'models.csv',index=False)
ceiling=np.mean([bool(set(ranks['04_hybrid'][i,:RERANK_TOP])&gold[i]) for i in t])
print('Hit ceiling kandidat rerank:',ceiling,'| metode final:',chosen)
audit=pd.DataFrame([{'query_id':i,'query':queries['query'].iloc[i],'top_doc':int(ranks[chosen][i,0]),
    'gold':sorted(gold[i]),**query_metrics(ranks[chosen][i],gold[i])} for i in t])
show('Kueri gagal',audit.nsmallest(10,'MRR@10'))
audit.to_csv(OUT/'query_results.csv',index=False)
print('Keterbatasan: default gold proxy satu artikel asal, bukan relevance judgment lengkap; ulangi pada kueri manusia sebelum klaim performa pencarian nyata.')


Test ranking


,method,MRR@10,Recall@1,Recall@5,Recall@10,NDCG@10
0,01_tfidf,0.9798,0.9708,0.9917,0.9958,0.9837
1,02_bm25,0.9910,0.9875,0.9958,0.9958,0.9922
2,03_e5_chunks,0.9561,0.9458,0.9667,0.9708,0.9597
3,04_hybrid,0.9910,0.9875,0.9958,0.9958,0.9922
4,05_hybrid_rerank,0.9742,0.9625,0.9875,0.9917,0.9785


Hit ceiling kandidat rerank: 0.9958333333333333 | metode final: 02_bm25

Kueri gagal


,query_id,query,top_doc,gold,MRR@10,Recall@1,Recall@5,Recall@10,NDCG@10
192,320,"Bek Manchester United , Marcos Rojo , terancam absen membela timnya sampai 2018 karena cedera lututnya tidak kunjung...",1059,[44],0.0000,0.0,0.0,0.0,0.0000
152,258,Hasil Seleksi Nasional Masuk Perguruan Tinggi Negeri ( SNMPTN ) diumumkan pada Rabu ( 26 / 4 ) .,1208,[1897],0.3333,0.0,1.0,1.0,0.5000
128,224,Kabar duka kembali mewarnai industri hiburan Tanah Air .,487,[1813],0.5000,0.0,1.0,1.0,0.6309
0,1,Hari jadi Kemerdekaan Republik Indonesia ( RI ) ke - 72 diwarnai duka .,353,[353],1.0000,1.0,1.0,1.0,1.0000
1,2,"Memasak makanan dengan ukuran normal saja sulit , apalagi jika membuat makanan dengan ukuran super kecil .",1333,[1333],1.0000,1.0,1.0,1.0,1.0000
2,3,"Tak hanya menginginkan bentuk bibir yang tipis , sebagian besar wanita menginginkan tubuh sempurna bak model Victori...",905,[905],1.0000,1.0,1.0,1.0,1.0000
3,7,Gula merupakan bahan makanan yang jamak digunakan untuk membuat suatu makanan atau minuman berasa manis .,1731,[1731],1.0000,1.0,1.0,1.0,1.0000
4,9,Keju keledai ternyata menjadi keju termahal di dunia .,1323,[1323],1.0000,1.0,1.0,1.0,1.0000
5,11,"Komisi Pemberantasan Korupsi akan memeriksa keponakan Ketua DPR Setya Novanto , Irvanto Hendra Pambudi Cahyo .",1292,[1292],1.0000,1.0,1.0,1.0,1.0000
6,12,Jennifer Lopez dan Marc Anthony menunjukkan bahwa pasangan yang sudah bercerai masih bisa disatukan oleh musik .,1118,[1118],1.0000,1.0,1.0,1.0,1.0000


Keterbatasan: default gold proxy satu artikel asal, bukan relevance judgment lengkap; ulangi pada kueri manusia sebelum klaim performa pencarian nyata.


## Interpretasi & sumber
Semua angka dihitung saat run; pemilihan validation tidak diubah setelah holdout. E5 memakai query: di kedua sisi STS, query:/passage: pada retrieval. [E5](https://huggingface.co/intfloat/multilingual-e5-base), [IndoBERT](https://huggingface.co/indobenchmark/indobert-base-p1), [IndoNLI](https://github.com/ir-nlp-csui/indonli).

In [14]:
import importlib.metadata as metadata
manifest={'task':RESULT_NAME,'seconds':time.perf_counter()-START,'gpu':torch.cuda.get_device_name(0),'seed':SEED,'versions':{p:metadata.version(p) for p in ['torch','transformers','scikit-learn','pandas']}}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print(manifest)

{'task': 'hasil_retrieval_review', 'seconds': 82.65204797599995, 'gpu': 'Tesla T4', 'seed': 42, 'versions': {'torch': '2.11.0+cu130', 'transformers': '5.20.2', 'scikit-learn': '1.6.1', 'pandas': '2.3.3'}}
